# Radar IB V1

Notebook inicial para explorar a base de oportunidades, testar a classificacao heuristica e validar o score decomposto.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.build_memo import generate_memo_outputs
from src.classify import classify_opportunities, load_taxonomy_config
from src.score import load_scoring_config, score_opportunities

RAW_FILE = PROJECT_ROOT / "data_raw" / "opps_raw.xlsx"
TAXONOMY_FILE = PROJECT_ROOT / "config" / "taxonomia.yaml"
SCORING_FILE = PROJECT_ROOT / "config" / "scoring.yaml"
PROCESSED_DIR = PROJECT_ROOT / "data_processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SCORED_OUTPUT = PROCESSED_DIR / "opps_scored.xlsx"
MEMO_QUEUE_OUTPUT = PROCESSED_DIR / "memo_queue.xlsx"
MEMO_OUTPUT = OUTPUTS_DIR / "memo_semana.md"

taxonomy = load_taxonomy_config(TAXONOMY_FILE)
scoring_config = load_scoring_config(SCORING_FILE)
RAW_FILE

In [ ]:
# Placeholder para a futura carga da base bruta.
# Exemplo:
# df = pd.read_excel(RAW_FILE)
# classified_df = classify_opportunities(df, taxonomy)
# scored_df = score_opportunities(classified_df, scoring_config)
# scored_df.to_excel(SCORED_OUTPUT, index=False)
# memo_queue_df, memo_markdown = generate_memo_outputs(
#     scored_df,
#     scoring_config=scoring_config,
#     queue_output_path=MEMO_QUEUE_OUTPUT,
#     memo_output_path=MEMO_OUTPUT,
# )
# memo_queue_df.head()

## Demonstracao rapida

Casos sinteticos para validar classificacao, score decomposto, prioridade e entrada no memo.

In [ ]:
demo_df = pd.DataFrame(
    [
        {
            "titulo": "Empresa de software avalia aquisicao de concorrente",
            "descricao": "Companhia abriu processo competitivo para compra de participacao.",
            "situacao_resumida": "Mandato em avaliacao com possivel fechamento no proximo trimestre.",
            "fonte": "Brazil Journal",
            "tipo_fonte": "midia especializada",
            "valor_divulgado": "R$ 650 mi",
            "data_evento": "2026-04-10",
            "data_captura": "2026-03-24"
        },
        {
            "titulo": "Industria apresenta plano de recuperacao judicial",
            "descricao": "Companhia protocolou pedido e plano de recuperacao para negociar com credores.",
            "situacao_resumida": "Discussao com bancos sobre waiver e venda de ativos.",
            "fonte": "Valor; Pipeline",
            "tipo_fonte": "multiplas fontes",
            "valor_divulgado": None,
            "data_evento": "2026-03-20",
            "data_captura": "2026-03-24"
        },
        {
            "titulo": "Companhia protocola pedido de IPO na CVM",
            "descricao": "Oferta publica inicial pode ocorrer ainda neste semestre.",
            "situacao_resumida": "Documentacao oficial ja em analise no regulador.",
            "fonte": "CVM",
            "tipo_fonte": "oficial",
            "valor_divulgado": "R$ 1,2 bi",
            "data_evento": "2026-04-15",
            "data_captura": "2026-03-24"
        },
        {
            "titulo": "Emissora prepara debentures para reforcar caixa",
            "descricao": "Operacao de emissao de divida pode sair nas proximas semanas.",
            "situacao_resumida": "Estrutura preliminar em discussao com bancos.",
            "fonte": "ANBIMA",
            "tipo_fonte": "oficial",
            "valor_divulgado": "R$ 300 mi",
            "data_evento": "2026-05-30",
            "data_captura": "2026-03-24"
        },
        {
            "titulo": "Grupo negocia bridge loan com fundo de capital estruturado",
            "descricao": "Transacao mira uma solucao de capital de curto prazo.",
            "situacao_resumida": "Processo bilateral e menos obvio para o mercado amplo.",
            "fonte": "Pipeline",
            "tipo_fonte": "midia especializada",
            "valor_divulgado": "R$ 180 mi",
            "data_evento": "2026-04-05",
            "data_captura": "2026-03-24"
        },
        {
            "titulo": "Governo prepara leilao de concessao rodoviaria",
            "descricao": "Projeto de infraestrutura deve atrair investidores para a rodovia.",
            "situacao_resumida": "Edital previsto para as proximas semanas.",
            "fonte": "ANTT",
            "tipo_fonte": "oficial",
            "valor_divulgado": "R$ 2,3 bi",
            "data_evento": "2026-04-18",
            "data_captura": "2026-03-24"
        }
    ]
)

classified_demo = classify_opportunities(demo_df, taxonomy)
classified_demo[["titulo", "fonte", "produto_trilha", "subtipo"]]

In [ ]:
scored_demo = score_opportunities(classified_demo, scoring_config)

score_columns = [
    "titulo",
    "produto_trilha",
    "subtipo",
    "score_materialidade",
    "just_materialidade",
    "score_mandatabilidade",
    "just_mandatabilidade",
    "score_timing",
    "just_timing",
    "score_qualidade_sinal",
    "just_qualidade_sinal",
    "score_aderencia",
    "just_aderencia",
    "score_competitividade",
    "just_competitividade",
    "score_total",
    "nivel_prioridade",
    "entra_memo",
]

scored_demo[score_columns]

## Fila editorial e memo semanal

Geracao dos artefatos editoriais a partir do DataFrame scored da demonstracao.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

scored_demo.to_excel(SCORED_OUTPUT, index=False)
memo_queue_demo, memo_markdown = generate_memo_outputs(
    scored_demo,
    scoring_config=scoring_config,
    queue_output_path=MEMO_QUEUE_OUTPUT,
    memo_output_path=MEMO_OUTPUT,
)

memo_columns = [
    "secao_memo",
    "titulo_memo",
    "produto_trilha_memo",
    "subtipo_memo",
    "score_total",
    "score_decomposto",
]

memo_queue_demo[memo_columns]

In [ ]:
print(memo_markdown)
MEMO_OUTPUT